In [ ]:
from pathlib import Path
W = Path("/kaggle/working"); W.mkdir(exist_ok=True)
(W / "analysis_gap_groups.py").write_text('"""Teacher-student knowledge-gap analysis (groups G1-G4) for E4 vs F1b checkpoints.\n\nInference only. No training, no hyper-parameter selection.\n\nGroups (teacher/student correctness on the SAME input view):\n  G1 teacher right, student right     G2 teacher right, student wrong\n  G3 teacher wrong, student right     G4 teacher wrong, student wrong\n\nPer-sample gradients are taken w.r.t. the STUDENT LOGITS (closed form, verified against autograd in\n`selftest`):\n  g_ce = softmax(s) - onehot(y)\n  g_kd = T * (softmax(s/T) - softmax(t/T))          # gradient of T^2 * KL(p_t^T || p_s^T)\ncos(g_ce, g_kd) > 0 means CE and KD push the student logits the same way; < 0 means they conflict.\nThe scalar KD mask (~0.5) and BC are not part of this analysis (BC gradient is ~1% of KD, see HANDOFF).\n\nUsage:\n  python analysis_gap_groups.py selftest\n  python analysis_gap_groups.py extract  OUT.npz --student CKPT --teacher CKPT --split train|test\n  python analysis_gap_groups.py report   --a E4=e4.npz --b F1b=f1b.npz --out report.json\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport importlib.util\nimport json\nimport os\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\nGROUPS = ("G1", "G2", "G3", "G4")\nT_DEFAULT = 4.0\nANALYSIS_SUBSET = 10000   # fixed subset of the 50k train images\nSUBSET_SEED = 0           # fixed, independent of any training seed\n\n\n# ----------------------------------------------------------------------------- pure-tensor core\ndef group_ids(t_logits: torch.Tensor, s_logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:\n    """0..3 for G1..G4. argmax is not differentiable, so no gradient can flow through the grouping."""\n    t_ok = t_logits.argmax(1) == y\n    s_ok = s_logits.argmax(1) == y\n    gid = torch.empty_like(y)\n    gid[t_ok & s_ok] = 0\n    gid[t_ok & ~s_ok] = 1\n    gid[~t_ok & s_ok] = 2\n    gid[~t_ok & ~s_ok] = 3\n    return gid\n\n\ndef per_sample_terms(s: torch.Tensor, t: torch.Tensor, y: torch.Tensor, T: float = T_DEFAULT) -> dict:\n    s, t = s.double(), t.double()\n    onehot = F.one_hot(y, s.size(1)).double()\n    ps, pt = F.softmax(s, 1), F.softmax(t, 1)\n    ce = F.cross_entropy(s, y, reduction="none")\n    ps_T, pt_T = F.softmax(s / T, 1), F.softmax(t / T, 1)\n    kd = (pt_T * (torch.log(pt_T.clamp_min(1e-12)) - torch.log(ps_T.clamp_min(1e-12)))).sum(1) * T * T\n    g_ce = ps - onehot\n    g_kd = T * (ps_T - pt_T)\n    cos = F.cosine_similarity(g_ce, g_kd, dim=1, eps=1e-12)\n    return {\n        "ce": ce, "kd": kd,\n        "p_t_y": pt.gather(1, y[:, None])[:, 0], "p_s_y": ps.gather(1, y[:, None])[:, 0],\n        "cos": cos, "n_ce": g_ce.norm(dim=1), "n_kd": g_kd.norm(dim=1),\n    }\n\n\ndef _boot_ci(x: np.ndarray, n_boot: int = 1000, seed: int = 0) -> tuple[float, float]:\n    if len(x) < 2:\n        return (float("nan"), float("nan"))\n    rng = np.random.default_rng(seed)\n    means = x[rng.integers(0, len(x), size=(n_boot, len(x)))].mean(1)\n    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))\n\n\ndef analyze(s: torch.Tensor, t: torch.Tensor, y: torch.Tensor, T: float = T_DEFAULT) -> dict:\n    gid = group_ids(t, s, y)\n    terms = per_sample_terms(s, t, y, T)\n    out = {"n_total": int(len(y)), "student_top1": float((s.argmax(1) == y).float().mean() * 100),\n           "teacher_top1": float((t.argmax(1) == y).float().mean() * 100), "groups": {}}\n    for k, name in enumerate(GROUPS):\n        m = gid == k\n        n = int(m.sum())\n        row = {"n": n, "share_pct": 100.0 * n / len(y)}\n        if n:\n            cos = terms["cos"][m].numpy()\n            lo, hi = _boot_ci(cos)\n            row.update({\n                "ce": float(terms["ce"][m].mean()), "kd": float(terms["kd"][m].mean()),\n                "p_t_y": float(terms["p_t_y"][m].mean()), "p_s_y": float(terms["p_s_y"][m].mean()),\n                "cos_mean": float(cos.mean()), "cos_median": float(np.median(cos)),\n                "cos_ci95": [lo, hi], "cos_neg_pct": float((cos < 0).mean() * 100),\n                "gnorm_ce": float(terms["n_ce"][m].mean()), "gnorm_kd": float(terms["n_kd"][m].mean()),\n                # per-sample cos is shared by sign; also report the norm-weighted share of KD in a 0.5/0.5 update\n                "kd_over_ce_norm": float((terms["n_kd"][m] / terms["n_ce"][m].clamp_min(1e-12)).median()),\n            })\n        out["groups"][name] = row\n    return out, gid\n\n\ndef transitions(gid_a: torch.Tensor, gid_b: torch.Tensor) -> dict:\n    mat = np.zeros((4, 4), dtype=int)\n    for a, b in zip(gid_a.tolist(), gid_b.tolist()):\n        mat[a, b] += 1\n    return {"rows=a_group, cols=b_group": mat.tolist()}\n\n\n# ----------------------------------------------------------------------------- model / data (needs CIFAR-100)\ndef _load_main_module():\n    os.environ.setdefault("ALLOW_CPU", "1")\n    path = Path(__file__).with_name("streamlined_kd_cifar100.py")\n    spec = importlib.util.spec_from_file_location("skd_main", path)\n    mod = importlib.util.module_from_spec(spec)\n    sys.modules["skd_main"] = mod\n    spec.loader.exec_module(mod)\n    return mod\n\n\n@torch.no_grad()\ndef _logits(model, loader, device):\n    model.eval()\n    outs = []\n    for x, _ in loader:\n        outs.append(model(x.to(device)).float().cpu())\n    return torch.cat(outs)\n\n\ndef extract(args) -> None:\n    import torchvision\n    import torchvision.transforms as transforms\n    from torch.utils.data import DataLoader, Subset\n\n    mod = _load_main_module()\n    device = mod.DEVICE\n    mod.prepare_cifar100()\n    norm = transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))\n    tf = transforms.Compose([transforms.ToTensor(), norm])   # deterministic, one view for both models\n    ds = torchvision.datasets.CIFAR100(mod.DATA_DIR, train=(args.split == "train"), download=False, transform=tf)\n    all_targets = np.asarray(ds.targets)                      # read once; never re-instantiate the dataset per sample\n    if args.split == "train":\n        idx = np.sort(np.random.default_rng(SUBSET_SEED).permutation(len(ds))[:ANALYSIS_SUBSET])\n        ds = Subset(ds, idx.tolist())\n    else:\n        idx = np.arange(len(ds))\n    labels = torch.from_numpy(all_targets[idx]).long()\n    loader = DataLoader(ds, batch_size=500, shuffle=False, num_workers=0)\n\n    teacher = mod.resnet56().to(device)\n    t_state = mod.load_state_dict(Path(args.teacher))\n    for key, value in teacher.state_dict().items():   # same handling as run_experiment: BN step counters only\n        if key.endswith("num_batches_tracked") and key not in t_state:\n            t_state[key] = torch.zeros_like(value)\n    teacher.load_state_dict(t_state, strict=True)\n    student = mod.resnet20().to(device)\n    raw = mod._torch_load(Path(args.student))\n    if isinstance(raw, dict) and "student" in raw:     # last_checkpoint.pt layout\n        raw = raw["student"]\n    student.load_state_dict(raw, strict=True)\n    s, t = _logits(student, loader, device), _logits(teacher, loader, device)\n    np.savez(args.out, s=s.numpy(), t=t.numpy(), y=labels.numpy(), idx=np.asarray(idx),\n             split=args.split, student=str(args.student), teacher=str(args.teacher))\n    print(f"saved {args.out}: n={len(labels)} student_top1={(s.argmax(1)==labels).float().mean()*100:.2f} "\n          f"teacher_top1={(t.argmax(1)==labels).float().mean()*100:.2f}")\n\n\ndef extract_aug(args) -> None:\n    """Training-distribution analysis: weak view (crop+flip) and strong view (crop+flip+RandAugment) of the SAME fixed\n    10k train subset. Augmented tensors are generated ONCE (fixed seed) and every model sees exactly the same tensors."""\n    import random\n    import torchvision\n    import torchvision.transforms as transforms\n\n    mod = _load_main_module()\n    device = mod.DEVICE\n    mod.prepare_cifar100()\n    norm = transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))\n    weak = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),\n                               transforms.ToTensor(), norm])\n    strong = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),\n                                 mod.PaperRandAugment(2), transforms.ToTensor(), norm])\n    base = torchvision.datasets.CIFAR100(mod.DATA_DIR, train=True, download=False)       # PIL images, no transform\n    idx = np.sort(np.random.default_rng(SUBSET_SEED).permutation(len(base))[:ANALYSIS_SUBSET])\n    y = torch.from_numpy(np.asarray(base.targets)[idx]).long()\n    random.seed(args.aug_seed); np.random.seed(args.aug_seed); torch.manual_seed(args.aug_seed)\n    xs_weak = torch.stack([weak(base[i][0]) for i in idx])\n    xs_strong = torch.stack([strong(base[i][0]) for i in idx])\n    x_all = torch.cat([xs_weak, xs_strong])\n    view = torch.cat([torch.zeros(len(idx)), torch.ones(len(idx))]).long()\n    y_all = torch.cat([y, y])\n\n    @torch.no_grad()\n    def run(model):\n        model.eval()\n        return torch.cat([model(x_all[i:i + 500].to(device)).float().cpu() for i in range(0, len(x_all), 500)])\n\n    teacher = mod.resnet56().to(device)\n    t_state = mod.load_state_dict(Path(args.teacher))\n    for key, value in teacher.state_dict().items():\n        if key.endswith("num_batches_tracked") and key not in t_state:\n            t_state[key] = torch.zeros_like(value)\n    teacher.load_state_dict(t_state, strict=True)\n    t = run(teacher)\n    for v, name in ((0, "weak"), (1, "strong")):\n        print(f"teacher top1 on {name} view: {(t[view == v].argmax(1) == y_all[view == v]).float().mean() * 100:.2f}")\n    Path(args.out_dir).mkdir(parents=True, exist_ok=True)\n    for spec in args.student:\n        name, path = spec.split("=", 1)\n        student = mod.resnet20().to(device)\n        raw = mod._torch_load(Path(path))\n        if isinstance(raw, dict) and "student" in raw:\n            raw = raw["student"]\n        student.load_state_dict(raw, strict=True)\n        sl = run(student)\n        out = Path(args.out_dir) / f"{name}.npz"\n        np.savez(out, s=sl.numpy(), t=t.numpy(), y=y_all.numpy(), view=view.numpy(), idx=np.asarray(idx))\n        print(f"saved {out}: " + "  ".join(\n            f"{vn} student_top1={(sl[view == v].argmax(1) == y_all[view == v]).float().mean() * 100:.2f}"\n            for v, vn in ((0, "weak"), (1, "strong"))))\n\n\ndef d0_cross_view(s_strong, t_strong, t_weak, y, T: float = T_DEFAULT) -> dict:\n    """Same student strong logits; KD target = teacher on the strong view (original) vs teacher on the paired weak view."""\n    orig = per_sample_terms(s_strong, t_strong, y, T)\n    cross = per_sample_terms(s_strong, t_weak, y, T)\n    pt_s, pt_w = F.softmax(t_strong.double(), 1), F.softmax(t_weak.double(), 1)\n    ps_s = F.softmax(s_strong.double(), 1)\n    tw_ok, ts_ok = t_weak.argmax(1) == y, t_strong.argmax(1) == y\n    m = 0.5 * (pt_s + pt_w)\n    js = 0.5 * ((pt_s * (pt_s.clamp_min(1e-12) / m).log()).sum(1) + (pt_w * (pt_w.clamp_min(1e-12) / m).log()).sum(1))\n    def blk(t):\n        c = t["cos"].numpy()\n        return {"conflict_pct": float((c < 0).mean() * 100), "cos_mean": float(c.mean()), "cos_ci95": list(_boot_ci(c)),\n                "kd_mean": float(t["kd"].mean()), "gnorm_kd_mean": float(t["n_kd"].mean()),\n                "gnorm_kd_p99": float(np.percentile(t["n_kd"].numpy(), 99)), "gnorm_ce_mean": float(t["n_ce"].mean()),\n                "finite": bool(torch.isfinite(t["kd"]).all() and torch.isfinite(t["n_kd"]).all())}\n    out = {"n": int(len(y)), "original": blk(orig), "cross": blk(cross),\n           "teacher_weak_acc": float(tw_ok.float().mean() * 100), "teacher_strong_acc": float(ts_ok.float().mean() * 100),\n           "teacher_weak_conf_y": float(pt_w.gather(1, y[:, None]).mean()), "teacher_strong_conf_y": float(pt_s.gather(1, y[:, None]).mean()),\n           "teacher_weak_maxprob": float(pt_w.max(1).values.mean()), "teacher_strong_maxprob": float(pt_s.max(1).values.mean()),\n           "tweak_right_tstrong_wrong_pct": float((tw_ok & ~ts_ok).float().mean() * 100),\n           "tweak_wrong_tstrong_right_pct": float((~tw_ok & ts_ok).float().mean() * 100),\n           "teacher_argmax_agree_pct": float((t_weak.argmax(1) == t_strong.argmax(1)).float().mean() * 100),\n           "js_teacher_weak_strong_mean": float(js.mean()),\n           "student_strong_acc": float((s_strong.argmax(1) == y).float().mean() * 100),\n           "student_strong_agree_tweak_pct": float((s_strong.argmax(1) == t_weak.argmax(1)).float().mean() * 100),\n           "semantic_lost_pct": float(((pt_s.gather(1, y[:, None])[:, 0] < 0.01) & (ps_s.gather(1, y[:, None])[:, 0] < 0.01)).float().mean() * 100),\n           "by_teacher_case": {}}\n    for name, m_ in (("Tw_right&Ts_right", tw_ok & ts_ok), ("Tw_right&Ts_wrong", tw_ok & ~ts_ok),\n                     ("Tw_wrong&Ts_right", ~tw_ok & ts_ok), ("Tw_wrong&Ts_wrong", ~tw_ok & ~ts_ok)):\n        n = int(m_.sum())\n        row = {"n": n, "pct": 100.0 * n / len(y)}\n        if n:\n            for tag, t in (("orig", orig), ("cross", cross)):\n                c = t["cos"][m_].numpy()\n                row[f"{tag}_conflict_pct"] = float((c < 0).mean() * 100)\n                row[f"{tag}_cos_mean"] = float(c.mean())\n                row[f"{tag}_gnorm_kd"] = float(t["n_kd"][m_].mean())\n        out["by_teacher_case"][name] = row\n    o, c = out["original"], out["cross"]\n    r_mean, r_p99 = c["gnorm_kd_mean"] / o["gnorm_kd_mean"], c["gnorm_kd_p99"] / o["gnorm_kd_p99"]\n    crit = {\n        "C1_conflict": c["conflict_pct"] <= 0.75 * o["conflict_pct"] and (o["conflict_pct"] - c["conflict_pct"]) >= 10.0,\n        "C2_gnorm": 0.67 <= r_mean <= 1.5 and 0.67 <= r_p99 <= 1.5 and o["finite"] and c["finite"],\n        "C3_pairing": out["teacher_weak_acc"] >= 95.0,   # label/idx equality is asserted by the caller\n        "C4_semantic": out["semantic_lost_pct"] <= 25.0,\n    }\n    out["gnorm_kd_ratio_mean"], out["gnorm_kd_ratio_p99"] = r_mean, r_p99\n    out["criteria"] = crit\n    out["decision"] = "PASS" if all(crit.values()) else "FAIL"\n    return out\n\n\ndef d0(args) -> None:\n    results = {}\n    for spec in args.npz:\n        name, path = spec.split("=", 1)\n        z = np.load(path, allow_pickle=True)\n        view = z["view"]\n        w, st = np.where(view == 0)[0], np.where(view == 1)[0]\n        assert len(w) == len(st), "unequal weak/strong counts"\n        # pairing: row k of the weak block and row k of the strong block come from the same original train image\n        idx = z["idx"]\n        assert len(idx) == len(w), "idx length must equal the number of original images"\n        assert np.array_equal(z["y"][w], z["y"][st]), "PAIRING FAILURE: labels differ between paired weak/strong rows"\n        y = torch.from_numpy(z["y"][st]).long()\n        r = d0_cross_view(torch.from_numpy(z["s"][st]), torch.from_numpy(z["t"][st]), torch.from_numpy(z["t"][w]), y)\n        r["pairing_labels_identical"] = True\n        results[name] = r\n        o, c = r["original"], r["cross"]\n        print(f"\\n===== D0 [{name}]  n_strong={r[\'n\']}  decision={r[\'decision\']}")\n        print(f"teacher acc weak {r[\'teacher_weak_acc\']:.2f} / strong {r[\'teacher_strong_acc\']:.2f};  p_T[y] weak {r[\'teacher_weak_conf_y\']:.3f} / strong {r[\'teacher_strong_conf_y\']:.3f};"\n              f"  maxprob weak {r[\'teacher_weak_maxprob\']:.3f} / strong {r[\'teacher_strong_maxprob\']:.3f}")\n        print(f"Tweak right & Tstrong wrong {r[\'tweak_right_tstrong_wrong_pct\']:.2f}%;  Tweak wrong & Tstrong right {r[\'tweak_wrong_tstrong_right_pct\']:.2f}%;"\n              f"  teacher argmax agree {r[\'teacher_argmax_agree_pct\']:.2f}%;  JS(Tw,Ts) {r[\'js_teacher_weak_strong_mean\']:.3f}")\n        print(f"student strong acc {r[\'student_strong_acc\']:.2f};  S_strong argmax == T_weak argmax {r[\'student_strong_agree_tweak_pct\']:.2f}%;  semantic-lost {r[\'semantic_lost_pct\']:.2f}%")\n        print("            conflict%   cos_mean [CI95]            KD      |g_KD| mean  |g_KD| p99  |g_CE| mean")\n        for tag, b in (("original", o), ("cross", c)):\n            print(f"{tag:10s}  {b[\'conflict_pct\']:7.2f}   {b[\'cos_mean\']:+.3f} [{b[\'cos_ci95\'][0]:+.3f},{b[\'cos_ci95\'][1]:+.3f}]  {b[\'kd_mean\']:6.3f}   {b[\'gnorm_kd_mean\']:.3f}      {b[\'gnorm_kd_p99\']:.3f}      {b[\'gnorm_ce_mean\']:.3f}")\n        print(f"|g_KD| ratio cross/original: mean {r[\'gnorm_kd_ratio_mean\']:.3f}, p99 {r[\'gnorm_kd_ratio_p99\']:.3f}")\n        for k, v in r["by_teacher_case"].items():\n            extra = "" if not v["n"] else (f"  conflict orig {v[\'orig_conflict_pct\']:.1f}% -> cross {v[\'cross_conflict_pct\']:.1f}%"\n                                         f"  cos {v[\'orig_cos_mean\']:+.3f} -> {v[\'cross_cos_mean\']:+.3f}  |g_KD| {v[\'orig_gnorm_kd\']:.3f} -> {v[\'cross_gnorm_kd\']:.3f}")\n            print(f"  {k:18s} n={v[\'n\']:5d} ({v[\'pct\']:5.2f}%){extra}")\n        print("criteria:", r["criteria"])\n    Path(args.out).write_text(json.dumps(results, indent=2))\n\n\ndef report(args) -> None:\n    def load(spec):\n        name, path = spec.split("=", 1)\n        z = np.load(path, allow_pickle=True)\n        keep = np.ones(len(z["y"]), dtype=bool)\n        if getattr(args, "view", "all") != "all":\n            keep = z["view"] == {"weak": 0, "strong": 1}[args.view]\n        return (name, torch.from_numpy(z["s"][keep]), torch.from_numpy(z["t"][keep]),\n                torch.from_numpy(z["y"][keep]).long())\n\n    (na, sa, ta, ya), (nb, sb, tb, yb) = load(args.a), load(args.b)\n    if not torch.equal(ya, yb) or not torch.allclose(ta, tb, atol=1e-4):\n        raise SystemExit("A and B must use identical samples/teacher logits (same split, same teacher)")\n    ra, ga = analyze(sa, ta, ya)\n    rb, gb = analyze(sb, tb, yb)\n    result = {na: ra, nb: rb, "transitions": transitions(ga, gb)}\n    Path(args.out).write_text(json.dumps(result, indent=2))\n    for name, r in ((na, ra), (nb, rb)):\n        print(f"\\n{name}: student {r[\'student_top1\']:.2f}  teacher {r[\'teacher_top1\']:.2f}  n={r[\'n_total\']}")\n        print("grp   n   share%   CE     KD    p_t[y] p_s[y]  cos_mean [95%CI]       cos<0%  |g_ce| |g_kd|")\n        for g in GROUPS:\n            x = r["groups"][g]\n            if x["n"]:\n                print(f"{g} {x[\'n\']:5d} {x[\'share_pct\']:6.2f} {x[\'ce\']:6.3f} {x[\'kd\']:6.3f} {x[\'p_t_y\']:6.3f} "\n                      f"{x[\'p_s_y\']:6.3f}  {x[\'cos_mean\']:+.3f} [{x[\'cos_ci95\'][0]:+.3f},{x[\'cos_ci95\'][1]:+.3f}] "\n                      f"{x[\'cos_neg_pct\']:6.1f} {x[\'gnorm_ce\']:6.3f} {x[\'gnorm_kd\']:6.3f}")\n    print("\\ntransitions (rows=%s group, cols=%s group):" % (na, nb))\n    print(np.array(result["transitions"]["rows=a_group, cols=b_group"]))\n\n\n# ----------------------------------------------------------------------------- self-test (synthetic, no data needed)\ndef selftest() -> None:\n    torch.manual_seed(0)\n    B, C, T = 64, 100, T_DEFAULT\n    s = (torch.randn(B, C) * 2).double().requires_grad_(True)\n    t = (torch.randn(B, C) * 3).double()\n    y = torch.randint(0, C, (B,))\n    terms = per_sample_terms(s.detach(), t, y, T)\n    # closed-form gradients vs autograd, per sample (loss sum => gradient row i belongs to sample i)\n    ce = F.cross_entropy(s, y, reduction="sum")\n    (g_ce_auto,) = torch.autograd.grad(ce, s)\n    kd = (T * T) * F.kl_div(F.log_softmax(s / T, 1), F.softmax(t / T, 1), reduction="sum")\n    (g_kd_auto,) = torch.autograd.grad(kd, s)\n    ps, pt = F.softmax(s.detach(), 1), F.softmax(t, 1)\n    g_ce = ps - F.one_hot(y, C).double()\n    g_kd = T * (F.softmax(s.detach() / T, 1) - F.softmax(t / T, 1))\n    assert torch.allclose(g_ce, g_ce_auto, atol=1e-10), "CE grad mismatch"\n    assert torch.allclose(g_kd, g_kd_auto, atol=1e-10), "KD grad mismatch"\n    assert torch.allclose(terms["kd"].sum(), kd.detach(), atol=1e-8), "KD value mismatch with F.kl_div*T^2"\n    assert torch.allclose(terms["cos"], F.cosine_similarity(g_ce_auto, g_kd_auto, dim=1), atol=1e-10)\n    # group partition is exhaustive and consistent with argmax\n    gid = group_ids(t, s.detach(), y)\n    assert gid.min() >= 0 and gid.max() <= 3\n    t_ok, s_ok = t.argmax(1) == y, s.detach().argmax(1) == y\n    assert torch.equal(gid == 1, t_ok & ~s_ok) and torch.equal(gid == 2, ~t_ok & s_ok)\n    # identical student/teacher => KD = 0 and g_kd = 0\n    z = per_sample_terms(t, t, y, T)\n    assert z["kd"].abs().max() < 1e-9 and z["n_kd"].max() < 1e-9\n    # kd closed-form matches repo kd_loss when importable (CPU)\n    try:\n        mod = _load_main_module()\n        ref = mod.kd_loss(s.detach().float(), t.float(), T, per_sample=True)\n        assert torch.allclose(ref.double(), terms["kd"], atol=1e-3), "differs from repo kd_loss"\n        print("repo kd_loss(per_sample) agrees with analysis KD")\n    except Exception as exc:  # repo kd_loss may require config; report, do not hide\n        print(f"[note] could not cross-check repo kd_loss: {type(exc).__name__}: {exc}")\n    r, _ = analyze(s.detach().float(), t.float(), y)\n    assert sum(r["groups"][g]["n"] for g in GROUPS) == B\n    print("selftest OK (synthetic logits only; NOT results from E4/F1b)")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    sub = ap.add_subparsers(dest="cmd", required=True)\n    sub.add_parser("selftest")\n    e = sub.add_parser("extract")\n    e.add_argument("out"); e.add_argument("--student", required=True); e.add_argument("--teacher", required=True)\n    e.add_argument("--split", choices=("train", "test"), default="train")\n    g = sub.add_parser("extract_aug")\n    g.add_argument("out_dir"); g.add_argument("--teacher", required=True)\n    g.add_argument("--student", action="append", required=True, help="NAME=PATH (repeatable)")\n    g.add_argument("--aug_seed", type=int, default=12345)\n    dd = sub.add_parser("d0")\n    dd.add_argument("--npz", action="append", required=True, help="NAME=PATH of an extract_aug npz (repeatable)")\n    dd.add_argument("--out", required=True)\n    r = sub.add_parser("report")\n    r.add_argument("--a", required=True); r.add_argument("--b", required=True); r.add_argument("--out", required=True)\n    r.add_argument("--view", choices=("all", "weak", "strong"), default="all")\n    a = ap.parse_args()\n    {"selftest": lambda: selftest(), "extract": lambda: extract(a), "extract_aug": lambda: extract_aug(a),\n     "report": lambda: report(a), "d0": lambda: d0(a)}[a.cmd]()\n')
print("script written")


In [ ]:
import glob, subprocess, sys
def find(name):
    hits = [p for p in glob.glob("/kaggle/input/**/" + name, recursive=True) if "gap-groups-aug" in p]
    assert len(hits) == 1, (name, hits)
    return hits[0]
E4_LAST, E4_BEST = find("aug/e4_last.npz"), find("aug/e4_best.npz")
print(E4_LAST, E4_BEST, sep="\n")
r = subprocess.run([sys.executable, "/kaggle/working/analysis_gap_groups.py", "d0", "--npz", f"E4_last={E4_LAST}",
                    "--npz", f"E4_best={E4_BEST}", "--out", "/kaggle/working/d0_cross_view.json"],
                   cwd="/kaggle/working", capture_output=True, text=True)
print(r.stdout[-12000:])
if r.returncode: print(r.stderr[-4000:]); raise SystemExit(r.returncode)
